In [25]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
from plotly.offline import iplot, init_notebook_mode

# Load the data
sup_df = pd.read_csv('../data/supplier_feed_clean.csv')
prod_df = pd.read_csv('../data/product_metadata.csv')

### Average Cost Price per Product Category

In [38]:
# Connect to the SQL database
connection = sqlite3.connect('../data/auto_parts.db')

query1 = """SELECT
    t2.category,
    AVG(t1.cost_price) AS average_price_per_category
FROM
    cleaned_supplier_data AS t1
INNER JOIN
    product_metadata AS t2
ON
    t1.part_id = t2.part_id
GROUP BY
    t2.category;
"""

average_prices_df = pd.read_sql_query(query1, connection)

print(average_prices_df)

# Close connection
connection.close()

      category  average_price_per_category
0       Brakes                  229.615504
1  Electronics                  227.151016
2       Engine                  228.901078
3      Exhaust                  224.624252
4      Filters                  229.501427
5         HVAC                  226.774780
6   Suspension                  233.024786


### Top 5 Parts with Highest Current Stock Levels

In [29]:
# Connect to the SQL database
connection = sqlite3.connect('../data/auto_parts.db')

query2 = """SELECT
    part_name,
    stock_level
FROM 
    (
        SELECT
            t2.part_name,
            t1.stock_level,
            ROW_NUMBER() OVER (
                PARTITION BY t1.part_id
                ORDER BY t1.entry_date DESC
            ) as row_number
        FROM
            cleaned_supplier_data AS t1
        INNER JOIN
            product_metadata AS t2
        ON
            t1.part_id = t2.part_id
    ) AS latest_data
WHERE
    row_number = 1
    AND stock_level != -1
ORDER BY
    stock_level DESC
LIMIT 5;
"""

highest_stock_levels = pd.read_sql_query(query2, connection)

print(highest_stock_levels)

# Close connection
connection.close()

            part_name  stock_level
0  Electronic Pad Set          493
1         HVAC Sensor          492
2    Electronic Rotor          492
3      Supenion Rotor          490
4        Filter Rotor          489


### Time Evolution of Supplier Entries

In [21]:
# Connect to the SQL database
connection = sqlite3.connect('../data/auto_parts.db')

query3 = """SELECT
    STRFTIME('%Y-%m', entry_date) AS year_month,
    COUNT(stock_entry_id) AS total_entries
FROM
    cleaned_supplier_data
GROUP BY
    year_month
ORDER BY
    year_month ASC;
"""

entry_frequency = pd.read_sql_query(query3, connection)

print(entry_frequency)

# Close connection
connection.close()

   year_month  total_entries
0     2024-01           1593
1     2024-02           1478
2     2024-03           1607
3     2024-04           1546
4     2024-05           1594
5     2024-06           1590
6     2024-07           1612
7     2024-08           1677
8     2024-09           1555
9     2024-10           1647
10    2024-11           1578
11    2024-12           1614
12    2025-01           1565
13    2025-02           1395
14    2025-03           1647
15    2025-04           1559
16    2025-05           1630
17    2025-06           1536
18    2025-07           1577


## Visualizations

### Average Cost Price per Product Category

In [ ]:
average_prices_df = average_prices_df.sort_values(
    by='average_price_per_category',
    ascending=True
)

fig = px.bar(
    average_prices_df, 
    x='category', 
    y='average_price_per_category', 
    title='Average Cost Price per Product Category',
    labels={'category': 'Category', 'average_price_per_category': 'Average price'}
)

fig.show()

### Top 5 Parts with Highest Current Stock Levels

In [48]:
highest_stock_levels = highest_stock_levels.sort_values(
    by='stock_level',
    ascending=False
)

fig = px.bar(
    highest_stock_levels, 
    x='part_name', 
    y='stock_level', 
    title='Top 5 Parts with Highest Current Stock Levels',
    labels={'part_name': 'Part name', 'stock_level': 'Stock level'}
)

fig.show()

### Time Evolution of Supplier Entries

In [ ]:
fig = px.line(
    entry_frequency, 
    x='year_month', 
    y='total_entries', 
    title='Time Evolution of Supplier Entries',
    labels={'year_month': 'Month', 'total_entries': 'Total number of entries'}
)

fig.show()